# Bronze to Silver Data Transformation (Sales)

All errors corrected in this notebook were identified in the bronze overview notebooks (Sales_2017_bronze, Sales_2018_bronze, Sales_2019_bronze, Sales_2020_bronze).

**Read files**

In [0]:
path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Sales_2017.csv"


sales_2017_uncleaned = (spark.read
      .format("csv")
      .option("header", "true")
      .option("sep", "\t")          
      .option("inferSchema", "true")
      .option("mode", "PERMISSIVE")
      .load(path))

display(sales_2017_uncleaned.limit(5))

In [0]:
path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Sales_2018.csv"


sales_2018_uncleaned = (spark.read
      .format("csv")
      .option("header", "true")
      .option("sep", "\t")          
      .option("inferSchema", "true")
      .option("mode", "PERMISSIVE")
      .load(path))

display(sales_2018_uncleaned.limit(5))

In [0]:
path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Sales_2019.csv"


sales_2019_uncleaned = (spark.read
      .format("csv")
      .option("header", "true")
      .option("sep", "\t")          
      .option("inferSchema", "true")
      .option("mode", "PERMISSIVE")
      .load(path))

display(sales_2019_uncleaned.limit(5))

In [0]:
path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Sales_2020.csv"


sales_2020_uncleaned = (spark.read
      .format("csv")
      .option("header", "true")
      .option("sep", "\t")          
      .option("inferSchema", "true")
      .option("mode", "PERMISSIVE")
      .load(path))

display(sales_2020_uncleaned.limit(5))

**Remove the extra zero from the ProductKeys in the Sales2017 DataFrame (data entry error)**

In [0]:
from pyspark.sql.functions import col, when

# Display the rows with four digit numbers in the ProductKey column
display(sales_2017_uncleaned.filter(col("ProductKey") > 999))

In [0]:
# Remove the extra 0
sales_2017_uncleaned = sales_2017_uncleaned.withColumn(
    "ProductKey",
    when(col("ProductKey") > 999, (col("ProductKey") / 10).cast("int"))
    .otherwise(col("ProductKey"))
)

In [0]:
display(sales_2017_uncleaned.filter(col("ProductKey") > 999))

**We union all the Sales tables into a single DataFrame.**

In [0]:
sales = (
    sales_2017_uncleaned
    .unionByName(sales_2018_uncleaned)
    .unionByName(sales_2019_uncleaned)
    .unionByName(sales_2020_uncleaned)

)

display(sales.limit(5))

In [0]:
sales.printSchema()

In [0]:
row_count = sales.count()
col_count = len(sales.columns)
print(f"number of rows: {row_count}, \nnumber of columns : {col_count}")

**Change Date Format**

In [0]:
from pyspark.sql.functions import regexp_replace, to_date

# Remove the day name from the OrderDate column
sales = sales.withColumn(
    "OrderDate",
    regexp_replace(col("OrderDate"), "^[A-Za-z]+, ", "")
)

# Convert the string column into the correct date format
sales = sales.withColumn(
    "OrderDate",
    to_date(col("OrderDate"), "MMMM d, yyyy")
)

display(sales.limit(5))

**Check for same SalesOrderNumber between the different dates** 

Each time a SalesOrderNumber appears it should have the same Order Date

In [0]:
from pyspark.sql.functions import countDistinct

display(sales.groupBy("SalesOrderNumber") \
    .agg(countDistinct("OrderDate").alias("distinct_dates")) \
    .filter("distinct_dates > 1"))

**Remove the "$" and the "," from the Unit Price, Cost and Sales columns**

In [0]:
from pyspark.sql.types import DecimalType

cols = ["Unit Price", "Cost", "Sales"]

for c in cols:
    sales = sales.withColumn(
        c,
        regexp_replace(col(c), "[$,]", "").cast(DecimalType(10,2))
    )

In [0]:
display(sales.limit(5))

In [0]:
display(sales)

**Change column names to lower case and snake case format**

In [0]:
sales = sales.withColumnRenamed("SalesOrderNumber", "sales_order_number") \
             .withColumnRenamed("OrderDate", "order_date") \
             .withColumnRenamed("ProductKey", "product_key") \
             .withColumnRenamed("ResellerKey", "reseller_key") \
             .withColumnRenamed("EmployeeKey", "employee_key") \
             .withColumnRenamed("SalesTerritoryKey", "sales_territory_key") \
             .withColumnRenamed("Quantity", "quantity") \
             .withColumnRenamed("Unit Price", "unit_price") \
             .withColumnRenamed("Cost", "cost") \
             .withColumnRenamed("Sales", "sales")

In [0]:
display(sales.limit(5))

**Check for data consistency**

Each time a SalesOrderNumber appears it should have the same ResellerKey , EmployeeKey and SalesTerritoryKey

In [0]:
display(sales.groupBy("sales_order_number") \
  .agg(countDistinct("reseller_key").alias("reseller_count")) \
  .filter("reseller_count > 1"))

In [0]:
display(sales.groupBy("sales_order_number")
.agg(countDistinct("employee_key").alias("reseller_count"))
.filter("reseller_count > 1"))

In [0]:
display(sales.groupBy("sales_order_number")
.agg(countDistinct("sales_territory_key").alias("reseller_count"))
.filter("reseller_count > 1"))

**Check for invalid values in columns quantiy, unit_price, sales and cost**

In [0]:
display(sales.filter(
    (sales.quantity <= 0) |
    (sales.unit_price <= 0) |
    (sales.sales <=0) |
    (sales.cost <= 0)
))

**Outlier Detection**

Outlier detection was performed per product because products differ not only in price ranges but also in the typical quantities purchased. Calculating thresholds per product ensures that sales values are evaluated relative to the normal behavior of each product.


In [0]:
product = spark.read.table("sales_lakehouse.silver.product")
display(product.limit(5))

Calculate outlier thresholds per product using the Interquartile Range (IQR) method

In [0]:
quantiles = sales.groupBy("product_key").agg(
    expr("percentile_approx(sales, 0.25)").alias("Q1"),
    expr("percentile_approx(sales, 0.75)").alias("Q3")
)

# Create IQR and upper bound columns
quantiles = quantiles.withColumn("IQR", (col("Q3") - col("Q1")))
quantiles = quantiles.withColumn(
    "upper_bound",
    # Use 3*IQR to detect only extreme outliers and avoid flagging legitimate high sales values
    col("Q3") + 3 * col("IQR") 
)

In [0]:
# Join the product with the quantiles dataframe to get the product column and make the results easier to interpret.
quantiles = product.select("product_key", "product").join(
    quantiles,
    on="product_key",
    how="inner"
)

display(quantiles.limit(5))

In [0]:
# Join the sales and quantiles dataframes
sales_quantiles = sales.select(
    "sales_order_number",
    "product_key",
    "unit_price",
    "quantity",
    "sales"
    ).join(
        quantiles.select("upper_bound", "product_key", "product"),
        on="product_key",
        how="inner"
)

display(sales_quantiles.limit(5))

In [0]:
# Find the rows where the sales was higher than the upper bound of the product (and therefore an outlier)
outliers = sales_quantiles.filter(
    (col("sales") > col('upper_bound'))
)

In [0]:
# Reorder columns
outliers = outliers.select(
    "sales_order_number",
    "product_key",
    "product",
    "unit_price",
    "quantity",
    "sales",
    "upper_bound")
display(outliers)

In [0]:
print(f"Outliers percentage is {outliers.count()/sales.count()*100}%")

Although statistical outliers may appear in the Sales variable, the dataset refers to orders placed by resellers. Large orders are therefore expected and may represent legitimate bulk purchases. For this reason, no outlier removal is performed.

**Create Delta Table Schema**

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS sales_lakehouse.silver.sales (
    sales_order_number string,
    order_date date,
    product_key int,
    reseller_key int,
    employee_key int,
    sales_territory_key int,
    quantity int,
    unit_price decimal(10,2),
    sales decimal(10,2),
    cost decimal(10,2)
)
USING DELTA
""")

In [0]:
sales.write.format("delta").mode("overwrite").saveAsTable("sales_lakehouse.silver.sales")

In [0]:

sales.count()

**Validation Query**

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM sales_lakehouse.silver.sales;

In [0]:
%sql
SELECT *
FROM sales_lakehouse.silver.sales;